# ADMET overlap audit for `MERGED_DATASET`

This notebook downloads and audits two property-data collections against `datasets/merged/MERGED_DATASET.parquet`:

- **TDC ADMET Group**: 22 endpoints with the official `train_val` and held-out `test` CSVs.
- **MoleculeNet complement**: ESOL, Lipophilicity, BBBP, ClinTox, SIDER, and Tox21 raw releases. These are intentionally reported as full releases because this notebook does not impose a new split on them.

The primary match is the full InChIKey derived with RDKit. Canonical SMILES are recorded for inspection, but are not the sole join key. The report also counts connectivity-only matches (the first InChIKey block); these are **candidates for manual review**, not automatic label joins, because they can differ in stereochemistry, salt form, tautomerism, or protonation.

The notebook never rewrites `MERGED_DATASET.parquet`. It writes downloaded property releases under `datasets/properties/` and audit outputs under `datasets/properties/admet_overlap/`.

## Requirements

Run with the repository's main NMR environment, which must provide `pandas`, `pyarrow`, and `RDKit`. The download uses only Python's standard library.

Primary sources: [TDC ADMET benchmark](https://tdcommons.ai/benchmark/admet_group/overview/) and the original MoleculeNet release links hosted by DeepChem.

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections import Counter
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import pandas as pd
import pyarrow.parquet as pq
from rdkit import Chem, RDLogger

# Invalid source structures are counted explicitly below. Silence RDKit parser
# diagnostics so a few malformed records do not flood notebook output.
RDLogger.DisableLog('rdApp.warning')
RDLogger.DisableLog('rdApp.error')

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'datasets').is_dir():
    raise RuntimeError('Run this notebook from the repository root.')

MERGED_DATASET_PATH = REPO_ROOT / 'datasets/merged/MERGED_DATASET.parquet'
PROPERTY_ROOT = REPO_ROOT / 'datasets/properties'
TDC_ROOT = PROPERTY_ROOT / 'tdc_admet'
TDC_ARCHIVE = TDC_ROOT / 'admet_group.zip'
TDC_GROUP_ROOT = TDC_ROOT / 'admet_group'
MOLECULENET_ROOT = PROPERTY_ROOT / 'moleculenet_admet'
REPORT_ROOT = PROPERTY_ROOT / 'admet_overlap'

# Set to True only when MERGED_DATASET.parquet has changed and the cached index
# needs rebuilding. The default reuses a cache only when source size and mtime match.
FORCE_REBUILD_MERGED_INDEX = True

TDC_ADMET_URL = 'https://dataverse.harvard.edu/api/access/datafile/4426004'
MOLECULENET_FILES = {
    'esol': {
        'filename': 'delaney-processed.csv',
        'url': 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv',
    },
    'lipophilicity': {
        'filename': 'Lipophilicity.csv',
        'url': 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/Lipophilicity.csv',
    },
    'bbbp': {
        'filename': 'BBBP.csv',
        'url': 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv',
    },
    'clintox': {
        'filename': 'clintox.csv.gz',
        'url': 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/clintox.csv.gz',
    },
    'sider': {
        'filename': 'sider.csv.gz',
        'url': 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/sider.csv.gz',
    },
    'tox21': {
        'filename': 'tox21.csv.gz',
        'url': 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz',
    },
}


def download_if_missing(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        print(f'Reusing {destination.relative_to(REPO_ROOT)}')
        return

    print(f'Downloading {url}')
    urlretrieve(url, destination)


def ensure_property_sources() -> None:
    if not TDC_GROUP_ROOT.is_dir():
        download_if_missing(TDC_ADMET_URL, TDC_ARCHIVE)
        print('Extracting the TDC ADMET archive')
        with ZipFile(TDC_ARCHIVE) as archive:
            archive.extractall(TDC_ROOT)

    for dataset, info in MOLECULENET_FILES.items():
        del dataset
        download_if_missing(info['url'], MOLECULENET_ROOT / info['filename'])


ensure_property_sources()

if not MERGED_DATASET_PATH.is_file():
    raise FileNotFoundError(f'Missing {MERGED_DATASET_PATH}')

print(f'MERGED_DATASET: {MERGED_DATASET_PATH}')
print(f'TDC endpoints found: {len(list(TDC_GROUP_ROOT.glob("*/test.csv")))}')
print(f'MoleculeNet releases found: {len(MOLECULENET_FILES)}')

In [ ]:
INDEX_CACHE = REPORT_ROOT / 'MERGED_DATASET_identity_index.parquet'
INDEX_METADATA = REPORT_ROOT / 'MERGED_DATASET_identity_index_metadata.json'


def structure_identifiers(smiles: object) -> dict[str, str | None]:
    if not isinstance(smiles, str) or not smiles.strip():
        return {
            'canonical_smiles_rdkit': None,
            'full_inchikey': None,
            'connectivity_inchikey': None,
        }

    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        return {
            'canonical_smiles_rdkit': None,
            'full_inchikey': None,
            'connectivity_inchikey': None,
        }

    canonical_smiles = Chem.MolToSmiles(
        molecule,
        canonical=True,
        isomericSmiles=True,
    )

    try:
        full_inchikey = Chem.MolToInchiKey(molecule)
    except Exception:
        full_inchikey = None

    connectivity_inchikey = (
        full_inchikey.split('-')[0]
        if full_inchikey
        else None
    )

    return {
        'canonical_smiles_rdkit': canonical_smiles,
        'full_inchikey': full_inchikey,
        'connectivity_inchikey': connectivity_inchikey,
    }


def merged_source_state(path: Path) -> dict[str, int]:
    stat = path.stat()
    return {
        'size_bytes': stat.st_size,
        'mtime_ns': stat.st_mtime_ns,
    }


def build_merged_identity_index(path: Path) -> tuple[pd.DataFrame, dict[str, int]]:
    parquet_file = pq.ParquetFile(path)
    if 'smiles_canonical' not in parquet_file.schema_arrow.names:
        raise KeyError('MERGED_DATASET has no smiles_canonical column')

    by_full_inchikey = {}
    invalid_or_unkeyed_rows = 0
    processed_rows = 0

    for batch in parquet_file.iter_batches(
        batch_size=10_000,
        columns=['smiles_canonical'],
    ):
        for smiles in batch.column(0).to_pylist():
            identifiers = structure_identifiers(smiles)
            full_inchikey = identifiers['full_inchikey']

            if full_inchikey is None:
                invalid_or_unkeyed_rows += 1
                continue

            row = by_full_inchikey.setdefault(
                full_inchikey,
                {
                    'full_inchikey': full_inchikey,
                    'connectivity_inchikey': identifiers['connectivity_inchikey'],
                    'nmr_smiles_canonical': identifiers['canonical_smiles_rdkit'],
                    'nmr_record_count': 0,
                },
            )
            row['nmr_record_count'] += 1

        processed_rows += batch.num_rows
        if processed_rows % 50_000 == 0:
            print(f'Processed {processed_rows:,} MERGED_DATASET records')

    index = pd.DataFrame(by_full_inchikey.values())
    summary = {
        'nmr_records': processed_rows,
        'nmr_invalid_or_unkeyed_rows': invalid_or_unkeyed_rows,
        'nmr_unique_molecules': len(index),
        'nmr_duplicate_records_beyond_unique_molecules': processed_rows - invalid_or_unkeyed_rows - len(index),
    }
    return index, summary


def load_or_build_merged_index() -> tuple[pd.DataFrame, dict[str, int]]:
    REPORT_ROOT.mkdir(parents=True, exist_ok=True)
    state = merged_source_state(MERGED_DATASET_PATH)

    if not FORCE_REBUILD_MERGED_INDEX and INDEX_CACHE.exists() and INDEX_METADATA.exists():
        cached_metadata = json.loads(INDEX_METADATA.read_text())
        if cached_metadata.get('merged_source_state') == state:
            print('Reusing cached MERGED_DATASET identity index')
            return pd.read_parquet(INDEX_CACHE), cached_metadata['summary']

    print('Building MERGED_DATASET identity index from Parquet batches')
    index, summary = build_merged_identity_index(MERGED_DATASET_PATH)
    index.to_parquet(INDEX_CACHE, index=False)
    INDEX_METADATA.write_text(json.dumps({
        'merged_source_state': state,
        'summary': summary,
    }, indent=2))
    return index, summary


nmr_index, nmr_summary = load_or_build_merged_index()
print(json.dumps(nmr_summary, indent=2))

In [ ]:
def annotate_property_structures(frame: pd.DataFrame, smiles_column: str) -> pd.DataFrame:
    result = frame.copy().reset_index(drop=True)
    result['_source_row'] = range(len(result))
    identifiers = [structure_identifiers(value) for value in result[smiles_column]]
    return pd.concat([result, pd.DataFrame(identifiers)], axis=1)


def collect_property_releases() -> list[dict[str, object]]:
    releases = []

    for test_path in sorted(TDC_GROUP_ROOT.glob('*/test.csv')):
        endpoint = test_path.parent.name
        for split_name, filename in [('test', 'test.csv'), ('train_val', 'train_val.csv')]:
            path = test_path.parent / filename
            releases.append({
                'source': 'TDC ADMET',
                'dataset': endpoint,
                'split': split_name,
                'path': path,
                'smiles_column': 'Drug',
            })

    for dataset, info in MOLECULENET_FILES.items():
        releases.append({
            'source': 'MoleculeNet',
            'dataset': dataset,
            'split': 'full_release_no_split',
            'path': MOLECULENET_ROOT / info['filename'],
            'smiles_column': 'smiles',
        })

    return releases


def audit_release(
    release: dict[str, object],
    merged_index: pd.DataFrame,
) -> tuple[dict[str, object], pd.DataFrame]:
    frame = pd.read_csv(release['path'])
    smiles_column = release['smiles_column']
    if smiles_column not in frame.columns:
        raise KeyError(f'{release["path"]} has no {smiles_column!r} column')

    properties = annotate_property_structures(frame, smiles_column)
    nmr_full = set(merged_index['full_inchikey'].dropna())
    nmr_connectivity = set(merged_index['connectivity_inchikey'].dropna())
    property_full = set(properties['full_inchikey'].dropna())
    property_connectivity = set(properties['connectivity_inchikey'].dropna())

    exact_full = nmr_full & property_full
    shared_connectivity = nmr_connectivity & property_connectivity
    exact_connectivity = set(
        merged_index.loc[
            merged_index['full_inchikey'].isin(exact_full),
            'connectivity_inchikey',
        ].dropna()
    )

    matched_nmr_records = int(
        merged_index.loc[
            merged_index['full_inchikey'].isin(exact_full),
            'nmr_record_count',
        ].sum()
    )

    matched_properties = properties[
        properties['full_inchikey'].isin(exact_full)
    ].copy()
    details = matched_properties.merge(
        merged_index,
        on=['full_inchikey', 'connectivity_inchikey'],
        how='left',
    )
    details.insert(0, 'source', release['source'])
    details.insert(1, 'dataset', release['dataset'])
    details.insert(2, 'split', release['split'])
    details.insert(3, 'property_file', str(Path(release['path']).relative_to(REPO_ROOT)))

    summary = {
        'source': release['source'],
        'dataset': release['dataset'],
        'split': release['split'],
        'property_rows': len(properties),
        'property_invalid_or_unkeyed_rows': int(properties['full_inchikey'].isna().sum()),
        'property_unique_molecules': len(property_full),
        'property_rows_beyond_unique_molecules': int(properties['full_inchikey'].notna().sum() - len(property_full)),
        'exact_overlap_unique_molecules': len(exact_full),
        'exact_overlap_property_percent': round(100 * len(exact_full) / len(property_full), 2) if property_full else 0.0,
        'MERGED_DATASET_records_with_exact_overlap': matched_nmr_records,
        'connectivity_overlap_groups': len(shared_connectivity),
        'connectivity_only_groups_without_full_match': len(shared_connectivity - exact_connectivity),
    }
    return summary, details


releases = collect_property_releases()
reports = []
match_tables = []

for release in releases:
    print(f'Auditing {release["source"]}: {release["dataset"]} ({release["split"]})')
    report, details = audit_release(release, nmr_index)
    reports.append(report)
    match_tables.append(details)

overlap_summary = pd.DataFrame(reports).sort_values(
    ['source', 'split', 'exact_overlap_unique_molecules', 'dataset'],
    ascending=[True, True, False, True],
).reset_index(drop=True)

exact_matches = pd.concat(match_tables, ignore_index=True)

REPORT_ROOT.mkdir(parents=True, exist_ok=True)
def make_arrow_safe(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for column in result.select_dtypes(include='object').columns:
        result[column] = result[column].astype('string')
    return result


overlap_summary.to_csv(REPORT_ROOT / 'overlap_summary.csv', index=False)
overlap_summary.to_parquet(REPORT_ROOT / 'overlap_summary.parquet', index=False)
exact_matches.to_csv(REPORT_ROOT / 'exact_matches.csv', index=False)
make_arrow_safe(exact_matches).to_parquet(
    REPORT_ROOT / 'exact_matches.parquet',
    index=False,
)

tdc_test_summary = overlap_summary.loc[
    (overlap_summary['source'] == 'TDC ADMET')
    & (overlap_summary['split'] == 'test')
].sort_values('exact_overlap_unique_molecules', ascending=False)

print('\nTDC ADMET: overlap with official test splits')
print(tdc_test_summary.to_string(index=False))
print('\nAll-source summary')
print(overlap_summary.to_string(index=False))
print(f'\nWrote reports to {REPORT_ROOT.relative_to(REPO_ROOT)}')

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


downloaded_files = [TDC_ARCHIVE]
downloaded_files.extend(
    MOLECULENET_ROOT / info['filename']
    for info in MOLECULENET_FILES.values()
)

manifest = {
    'MERGED_DATASET': {
        'path': str(MERGED_DATASET_PATH.relative_to(REPO_ROOT)),
        'identity_policy': 'RDKit full InChIKey primary; first InChIKey block reported only as a flagged connectivity fallback',
        **nmr_summary,
    },
    'sources': {
        'tdc_admet_url': TDC_ADMET_URL,
        'moleculenet': MOLECULENET_FILES,
    },
    'downloaded_files': [
        {
            'path': str(path.relative_to(REPO_ROOT)),
            'size_bytes': path.stat().st_size,
            'sha256': sha256(path),
        }
        for path in downloaded_files
    ],
}

manifest_path = REPORT_ROOT / 'provenance.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f'Wrote {manifest_path.relative_to(REPO_ROOT)}')

# Exact matches are the candidates suitable for a later, explicit
# molecule-level split. Do not use connectivity-only matches as labels.
exact_matches.head(20)